# ipaddress

On Python 3, `import ipaddress` always loads the standard-library module, because the standard library comes before site-packages on `sys.path`. The `ipaddress` distribution on PyPI is a backport of that module for Python 2. On Python 3 these cells therefore exercise the standard-library `ipaddress` module, not the PyPI package.

## Example 1: Creating and analyzing IPv4 addresses with IPv4Address()

In [ ]:
import ipaddress

# Example 1: Creating and analyzing IPv4 addresses with IPv4Address()
print("Example 1: IPv4 address creation and analysis")

# Create IPv4 addresses
ip_addresses = ['192.168.1.1', '8.8.8.8', '127.0.0.1', '10.0.0.1', '172.16.0.1']

print("Analyzing IPv4 addresses:")
for ip_str in ip_addresses:
    ip = ipaddress.IPv4Address(ip_str)
    print(f"  {ip}:")
    print(f"    Version: {ip.version}")
    print(f"    Is private: {ip.is_private}")
    print(f"    Is loopback: {ip.is_loopback}")
    print(f"    Is multicast: {ip.is_multicast}")
    print(f"    Packed: {ip.packed}")

# Test address arithmetic
print(f"\nAddress arithmetic:")
base_ip = ipaddress.IPv4Address('192.168.1.1')
next_ip = base_ip + 1
prev_ip = base_ip - 1
print(f"  {base_ip} + 1 = {next_ip}")
print(f"  {base_ip} - 1 = {prev_ip}")

assert next_ip == ipaddress.IPv4Address('192.168.1.2') and prev_ip == ipaddress.IPv4Address('192.168.1.0')
assert ipaddress.IPv4Address('192.168.1.1').is_private and not ipaddress.IPv4Address('8.8.8.8').is_private
assert ipaddress.IPv4Address('127.0.0.1').is_loopback
assert ipaddress.IPv4Address('224.0.0.1').is_multicast
assert ipaddress.IPv4Address('10.0.0.1').packed == bytes([10, 0, 0, 1])
assert int(ipaddress.IPv4Address('0.0.1.0')) == 256

## Example 2: Working with IPv4 networks using IPv4Network()

In [ ]:
import ipaddress

# Example 2: Working with IPv4 networks using IPv4Network()
print("Example 2: IPv4 network operations")

# Create IPv4 networks
networks = ['192.168.1.0/24', '10.0.0.0/8', '172.16.0.0/12', '203.0.113.0/24']

print("Analyzing IPv4 networks:")
for net_str in networks:
    network = ipaddress.IPv4Network(net_str)
    print(f"  {network}:")
    print(f"    Network address: {network.network_address}")
    print(f"    Broadcast address: {network.broadcast_address}")
    print(f"    Netmask: {network.netmask}")
    print(f"    Total addresses: {network.num_addresses}")
    print(f"    Is private: {network.is_private}")

# Test network membership
print(f"\nNetwork membership testing:")
test_network = ipaddress.IPv4Network('192.168.1.0/24')
test_addresses = ['192.168.1.50', '192.168.1.255', '192.168.2.1']

for addr_str in test_addresses:
    addr = ipaddress.IPv4Address(addr_str)
    is_member = addr in test_network
    print(f"  {addr} in {test_network}: {is_member}")

net = ipaddress.IPv4Network('192.168.1.0/24')
assert str(net.network_address) == '192.168.1.0' and str(net.broadcast_address) == '192.168.1.255'
assert str(net.netmask) == '255.255.255.0' and net.num_addresses == 256
assert ipaddress.IPv4Network('10.0.0.0/8').num_addresses == 2 ** 24
assert ipaddress.IPv4Address('192.168.1.50') in net and ipaddress.IPv4Address('192.168.2.1') not in net
# strict=True (the default) rejects host bits set in the network address
try:
    ipaddress.IPv4Network('192.168.1.1/24')
    raise AssertionError("expected ValueError")
except ValueError:
    pass
assert ipaddress.IPv4Network('192.168.1.1/24', strict=False) == net

## Example 3: Using ip_address() function for automatic detection

In [ ]:
import ipaddress

# Example 3: Using ip_address() function for automatic detection
print("Example 3: Automatic IP address detection with ip_address()")

# Test addresses - mix of IPv4 and IPv6
test_addresses = [
    '192.168.1.1',           # IPv4
    '8.8.8.8',               # IPv4 public
    '::1',                   # IPv6 loopback
    '2001:db8::1',           # IPv6
    'fe80::1',               # IPv6 link-local
    '127.0.0.1'              # IPv4 loopback
]

print("Auto-detecting IP address types:")
for addr_str in test_addresses:
    addr = ipaddress.ip_address(addr_str)
    print(f"  {addr_str} -> {addr}")
    print(f"    Type: {type(addr).__name__}")
    print(f"    Version: {addr.version}")
    print(f"    Is private: {addr.is_private}")
    print(f"    Is loopback: {addr.is_loopback}")

assert isinstance(ipaddress.ip_address('8.8.8.8'), ipaddress.IPv4Address)
assert isinstance(ipaddress.ip_address('2001:db8::1'), ipaddress.IPv6Address)
assert ipaddress.ip_address('::1').is_loopback and ipaddress.ip_address('fe80::1').is_link_local

# ip_address() raises ValueError for text that is neither IPv4 nor IPv6
for bad in ['256.1.1.1', 'not-an-ip', '2001:db8::g']:
    try:
        ipaddress.ip_address(bad)
        raise AssertionError(f"expected ValueError for {bad!r}")
    except ValueError:
        pass

# Test compressed representation
print(f"\nAddress representations:")
ipv6_addr = ipaddress.ip_address('2001:0db8:0000:0000:0000:0000:0000:0001')
print(f"  Full: {ipv6_addr.exploded}")
print(f"  Compressed: {ipv6_addr.compressed}")

assert ipv6_addr.compressed == '2001:db8::1'
assert ipv6_addr.exploded == '2001:0db8:0000:0000:0000:0000:0000:0001'

## Example 4: Network summarization with summarize_address_range()

In [ ]:
import ipaddress

# Example 4: Network summarization with summarize_address_range()
print("Example 4: Network summarization and address ranges")

# Create address ranges for summarization
start_addresses = [
    ipaddress.IPv4Address('192.168.1.1'),
    ipaddress.IPv4Address('10.0.0.1'),
    ipaddress.IPv4Address('172.16.0.1')
]

end_addresses = [
    ipaddress.IPv4Address('192.168.1.100'),
    ipaddress.IPv4Address('10.0.0.255'),
    ipaddress.IPv4Address('172.16.0.50')
]

print("Summarizing address ranges:")
for start_addr, end_addr in zip(start_addresses, end_addresses):
    print(f"  Range: {start_addr} - {end_addr}")
    
    # Summarize the range into minimal networks
    networks = list(ipaddress.summarize_address_range(start_addr, end_addr))
    print(f"    Summarized into {len(networks)} networks:")
    for network in networks:
        print(f"      {network} ({network.num_addresses} addresses)")

# Test with IPv6 range
print(f"\nIPv6 range summarization:")
ipv6_start = ipaddress.IPv6Address('2001:db8::1')
ipv6_end = ipaddress.IPv6Address('2001:db8::100')
ipv6_networks = list(ipaddress.summarize_address_range(ipv6_start, ipv6_end))
print(f"  IPv6 range: {ipv6_start} - {ipv6_end}")
print(f"  Networks: {len(ipv6_networks)} summarized networks")

# Each range is covered exactly, by the fewest CIDR blocks
for start_addr, end_addr in zip(start_addresses, end_addresses):
    networks = list(ipaddress.summarize_address_range(start_addr, end_addr))
    assert networks[0].network_address == start_addr and networks[-1].broadcast_address == end_addr
    assert sum(n.num_addresses for n in networks) == int(end_addr) - int(start_addr) + 1
assert [str(n) for n in ipaddress.summarize_address_range(
    ipaddress.IPv4Address('10.0.0.0'), ipaddress.IPv4Address('10.0.0.255'))] == ['10.0.0.0/24']
assert len(ipv6_networks) == 9

## Example 5: Network collapse with collapse_addresses()

In [ ]:
import ipaddress

# Example 5: Network collapse with collapse_addresses()
print("Example 5: Collapsing overlapping networks")

# Create overlapping and adjacent networks
networks = [
    ipaddress.IPv4Network('192.168.1.0/24'),
    ipaddress.IPv4Network('192.168.2.0/24'),
    ipaddress.IPv4Network('192.168.1.0/25'),    # Overlaps with first
    ipaddress.IPv4Network('192.168.1.128/25'),  # Adjacent to previous
    ipaddress.IPv4Network('10.0.0.0/24'),
    ipaddress.IPv4Network('10.0.1.0/24'),
    ipaddress.IPv4Network('172.16.0.0/16')
]

print("Original networks:")
for network in networks:
    print(f"  {network}")

# Collapse overlapping networks
collapsed = list(ipaddress.collapse_addresses(networks))

print(f"\nAfter collapsing:")
print(f"  Original: {len(networks)} networks")
print(f"  Collapsed: {len(collapsed)} networks")

for network in collapsed:
    print(f"  {network}")

# Test with IPv6 networks
print(f"\nIPv6 network collapse:")
ipv6_networks = [
    ipaddress.IPv6Network('2001:db8::/64'),
    ipaddress.IPv6Network('2001:db8:0:1::/64'),
    ipaddress.IPv6Network('2001:db8::/63')  # Superset of first two
]

ipv6_collapsed = list(ipaddress.collapse_addresses(ipv6_networks))
print(f"  IPv6 original: {len(ipv6_networks)}, collapsed: {len(ipv6_collapsed)}")

assert [str(n) for n in collapsed] == ['10.0.0.0/23', '172.16.0.0/16', '192.168.1.0/24', '192.168.2.0/24']
assert [str(n) for n in ipv6_collapsed] == ['2001:db8::/63']